In [3]:
# 1. Import libraries
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Load data
df = pd.read_csv("IMDB Dataset.csv")  # adjust filename after upload
print(df.head())
print(df['sentiment'].value_counts())  # check it's balanced

# 3. Clean the text — remove HTML tags, punctuation, lowercase everything
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags like <br />
    text = re.sub(r"[^a-z\s]", " ", text)      # keep only letters
    text = re.sub(r"\s+", " ", text).strip()   # collapse extra spaces
    return text

df['clean_review'] = df['review'].apply(clean_text)

# 4. Convert labels to 0/1
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 5. Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.2, random_state=42
)

# 6. TF-IDF vectorization — turns text into numeric features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 7. Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# 8. Evaluate
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# 9. Test on your own sentence — great for demoing the project
sample = ["This movie was absolutely wonderful, I loved every minute!"]
sample_clean = [clean_text(s) for s in sample]
sample_tfidf = vectorizer.transform(sample_clean)
print(model.predict(sample_tfidf))  # 1 = positive, 0 = negative

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Accuracy: 0.8884
              precision    recall  f1-score   support

           0       0.90      0.87      0.89      4961
           1       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

[[4323  638]
 [ 478 4561]]
[1]
